# 40 — Demo generation (agriculture example): synthetic fixtures and the V2 → V3 controlled update

This is the **demo/test notebook** for the default agriculture domain. It generates synthetic fixture files, runs one ingestion pass, asks the baseline question (8-day answer from V2), publishes V3 from a **different filename**, and asserts that the answer, citations, and active pool follow the new version (6-day answer).

Everything file-generating or adversarial lives in this notebook and in `45_robustness_checks` — the operational notebooks (`20_ingestion_publication`, `30_retrieval_answers`) never generate files or inject test fixtures.

**Execution order:** run top to bottom on a fresh (or reset) schema; see README for the full project order.

In [0]:
%run ./20_ingestion_publication

In [0]:
%run ./30_retrieval_answers

## Generate synthetic fixtures: V1/V2 land now; V3 remains staged for the update demo

Creates five fictional documents × three revisions. Filenames deliberately differ across revisions of the same `document_id` (`original_crop_*.md`, `revised_guidance_*.md`, `new_field_manual_*.md`) to prove the pipeline follows the embedded business key, never the filename. V1/V2 land in `landing/` now; V3 waits in `staged_updates/` for the controlled update below.

`make_document` assumes the default agriculture `DOMAIN_ATTRIBUTES` (`title`, `crop`, `region`, `topic`); retargeting to another domain requires writing a domain-appropriate fixture generator alongside the configuration change. All agriculture content is fictional demonstration data, not advice.

In [0]:
def make_document(number, version):
    """Build one synthetic demo document (JSON metadata header + Markdown body).

    The review interval differs per version (V1=10, V2=8, V3=6 days) and only
    V1 carries the retired LEGACY-RED marker, which makes any leak of retired
    content detectable in later tests.
    """
    crop, topic = [('wheat','irrigation'), ('maize','soil monitoring'),
                   ('rice','field monitoring'), ('tomato','greenhouse operations'),
                   ('apple','orchard maintenance')][number-1]
    days = {1:10, 2:8, 3:6}[version]
    doc_id = f'AGR-{number:03d}'
    meta = dict(document_id=doc_id, version_id=f'{doc_id}-V{version}',
                document_date={1:'2026-01-15',2:'2026-04-15',3:'2026-07-15'}[version],
                title=f'{crop.title()} {topic.title()} Demo Guide', crop=crop,
                region='demo-region', topic=topic)
    body = f'''# Scope
SYNTHETIC DEMO ONLY. This fictional document is not agricultural advice.
This guide concerns {crop} {topic} in demo-region.

# Review interval
The {topic} review interval for {crop} is {days} days.
Current evidence marker: AGR{number:03d}_V{version}_CURRENT.
Record the review date, responsible operator, and observations in the demonstration log.

# Escalation
If a demonstration sensor reading is missing, flag the record for manual review.
Do not infer a measurement from an earlier observation.
'''
    if version == 1:
        body += '\n# Retired-only instruction\nThe legacy escalation code is LEGACY-RED.\n'
    return '---metadata\n' + json.dumps(meta, indent=2) + '\n---\n' + body


def generate_samples():
    """Write the demo corpus once: V1/V2 into landing/, V3 into staged_updates/.

    Existing files are skipped, so fixture generation is idempotent and never
    overwrites a demo already in progress.
    """
    for number in range(1, 6):
        for version in (1, 2, 3):
            folder = STAGED if version == 3 else LANDING
            # Different filenames across revisions of the same business key, intentionally.
            name = {1:f"original_crop_{number}.md", 2:f"revised_guidance_{number}.md",
                    3:f"new_field_manual_{number}.md"}[version]
            path = folder / name
            if not path.exists(): path.write_text(make_document(number, version))

generate_samples()
print("Initial files:", len(list(LANDING.glob("*.md"))), "Staged updates:", len(list(STAGED.glob("*.md"))))

## Ingest V1/V2, publish V2 active, and ask the baseline question

On a fresh schema the grounded answer is **8 days** citing AGR-001-V2 (2026-04-15). When demonstrating the full RAG path, confirm the answer `mode` is `GROUNDED_LLM` and the audit row is `SEMANTIC`; an `EXTRACTIVE` or `TFIDF` label is a clearly marked fallback, not the full path.

In [0]:
q = 'What is the irrigation review interval for wheat?'
print(ingest_and_publish())
display(spark.table(table("document_manifest")).orderBy("document_id", "document_date"))
print(json.dumps(ask(q, 'AGR-001'), indent=2))
print('Embeddings enabled:', embedding_available, 'Chat enabled:', chat_available)

## Controlled V2 → V3 update

Demonstrates that the answer follows the business key, not the filename: V3 is published from `new_field_manual_1.md` (a different filename than V2's `revised_guidance_1.md`), and the same question flips from the 8-day V2 answer to a 6-day V3 answer. Expected results, asserted by the next cell:

* V3 becomes active; an explicitly captured V2 chunk candidate fails registry validation; no V1/V2 chunks remain in the active retrieval source.
* Re-running ingestion on unchanged files adds no rows (idempotency).
* Evidence cites only AGR-001-V3 and contains only the current 6-day interval.

New active-chunk embeddings are computed lazily on the next retrieval; no search-index synchronization is needed. Rerunning this schema preserves V3 — use a fresh schema (or `reset_project_data`) for a fresh before/after baseline.

In [0]:
# Controlled update demo: publish V3 (different filename, later business date)
# and verify the active version, citations, and answer all follow the new version.
baseline = registry_snapshot()['AGR-001']
# Capture a live V2 chunk now; after publication it must FAIL registry validation.
stale_candidate = next(r.asDict() for r in spark.table(table('document_chunks')).filter(
    (F.col('document_id')=='AGR-001') & (F.col('version_id')=='AGR-001-V2')).collect())
update = STAGED / 'new_field_manual_1.md'
(LANDING / update.name).write_text(update.read_text())
print(ingest_and_publish())
current = registry_snapshot()
assert current['AGR-001'][0] == 'AGR-001-V3'
assert validate_candidates([stale_candidate], current) == [], 'Stale version leaked'
assert spark.table(table('active_document_chunks')).filter(
    (F.col('document_id')=='AGR-001') & (F.col('version_id')!='AGR-001-V3')).count() == 0
# Idempotency check: re-ingesting the same unchanged files must add no rows.
counts_before = (spark.table(table('document_versions')).count(), spark.table(table('document_chunks')).count())
ingest_and_publish()
assert counts_before == (spark.table(table('document_versions')).count(), spark.table(table('document_chunks')).count())
print('PASS: V3 active, V2 rejected, rerun idempotent. Baseline:', baseline)
print(json.dumps(ask(q, 'AGR-001'), indent=2))

# The same question must now cite only V3 and the 6-day interval.
result = ask(q, 'AGR-001')
assert result['sources'] and all(s['version_id']=='AGR-001-V3' for s in result['sources'])
assert any('6 days' in e for e in result['evidence'])
assert not any('8 days' in e or '10 days' in e for e in result['evidence'])
print('PASS: current excerpts cite V3 and contain only the current review interval.')